In [ ]:
import sys
import os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)


In [ ]:
import torch
import random
import numpy as np
import csv

In [ ]:
from models.semcovert import create_network
from utils.video_utils import load_video_as_tensor,split_video_tensor, get_video_files

In [ ]:
def seet_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seet_seed(42)

In [ ]:
def load_videos(data_dir, target_size=(240,320), max_frames=5): 
    
    video_files = get_video_files(data_dir)
    if not video_files:
        raise ValueError(f"No video files found in {data_dir}")
    
    print(f"Found {len(video_files)} video files: {[f.name for f in video_files]}")
    
    all_video_batches = []
    for video_file in video_files:
        video_tensor = load_video_as_tensor(
            str(video_file), 
            target_size=target_size
        )
        video_tensor = split_video_tensor(video_tensor, max_frames)
        if video_tensor.numel() > 0:
            for i in range(video_tensor.shape[0]):
                all_video_batches.append(video_tensor[i])
            del video_tensor  
        else:
            print(f"  Failed to load video: {video_file}")
    
    if not all_video_batches:
        raise ValueError("No valid video data loaded")
    
    all_video_tensor = torch.stack(all_video_batches, dim=0)
    del all_video_batches

    if len(all_video_tensor) % 2 != 0:
        all_video_tensor = all_video_tensor[:-1] # Ensure even number of videos

    mid = len(all_video_tensor) // 2
    cover_videos = all_video_tensor[:mid]
    secret_videos = all_video_tensor[mid:]
    
    del all_video_tensor  
    print(f"Loaded {cover_videos.shape[0]} cover videos and secret videos")
    
    return cover_videos, secret_videos

In [ ]:
def generate(model, cover_video, secret_video, batch_size, device):
    recon_cover = torch.empty_like(cover_video)
    if secret_video is not None:
        recon_secret = torch.empty_like(secret_video) 
    else:
        recon_secret = None
    with torch.no_grad():
        for i in range(0, cover_video.shape[0], batch_size):
            batch_cover = cover_video[i:i + batch_size].to(device)
            if secret_video is not None:
                batch_secret = secret_video[i:i + batch_size].to(device) 
            else:
                batch_secret = None
   
            output = model(batch_cover, batch_secret)
            recon_cover_chunk = output['cover_reconstructed'].cpu()
            recon_cover[i:i + batch_size] = recon_cover_chunk
            if secret_video is not None:
                recon_secret_chunk = output['secret_reconstructed'].cpu()
                recon_secret[i:i + batch_size] = recon_secret_chunk

    return recon_cover, recon_secret

In [ ]:
model_cfg = {
        'depth': 4,
        'dim': 96,
        'use_channel': True,
        'vae_config': {
            'dim': 96,
            'z_dim': 16,
            'dim_mult': [1, 2, 4, 4],
            'num_res_blocks': 2,
            'attn_scales': [],
            'temperal_downsample': [False, True, True],
            'dropout': 0.0
        },
        'channel_config': {
            'channel_type': 'AWGN',  # Channel type
            'snr': 25,               # Signal-to-noise ratio
        },    
    }

In [ ]:

from utils.metrics import calculate_psnr, calculate_ssim, calculate_fvd
import torchvision.models.video as model_3d
from torchvision.models.video import S3D_Weights, R3D_18_Weights, Swin3D_T_Weights
import torch.nn as nn
MODEL1 = nn.Sequential(
    model_3d.s3d(weights=S3D_Weights.DEFAULT).features,
    nn.AdaptiveAvgPool3d((1, 1, 1)),
)
MODEL2 = nn.Sequential(
    model_3d.r3d_18(weights=R3D_18_Weights.DEFAULT).stem,
    model_3d.r3d_18(weights=R3D_18_Weights.DEFAULT).layer1,
    model_3d.r3d_18(weights=R3D_18_Weights.DEFAULT).layer2,
    model_3d.r3d_18(weights=R3D_18_Weights.DEFAULT).layer3,
    model_3d.r3d_18(weights=R3D_18_Weights.DEFAULT).layer4,
    nn.AdaptiveAvgPool3d((1, 1, 1)),
)
class model3(nn.Module):
    def __init__(self, model):
        super(model3, self).__init__()
        self.model = model

    def forward(self, x):
        x = self.model.patch_embed(x)  # B _T _H _W C
        x = self.model.pos_drop(x)
        x = self.model.features(x)  # B _T _H _W C
        x = self.model.norm(x)
        x = x.permute(0, 4, 1, 2, 3)  # B, C, _T, _H, _W
        x = self.model.avgpool(x)
        return x
    
MODEL3 = model3(model_3d.swin3d_t(weights=Swin3D_T_Weights.DEFAULT))

def calculate_metrics(cover, recon_cover):
    psnr = calculate_psnr(cover, recon_cover)
    ssim = calculate_ssim(cover, recon_cover)
    fvd1 = calculate_fvd(MODEL1, cover, recon_cover, batch_size=8)
    fvd2 = calculate_fvd(MODEL2, cover, recon_cover, batch_size=8)
    fvd3 = calculate_fvd(MODEL3, cover, recon_cover, batch_size=8)

    return psnr, ssim, fvd1, fvd2, fvd3

In [ ]:
model_path = "YOUR_MODEL_PATH"  # Path to your model weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_network(model_cfg)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

In [ ]:
csv_file = "YOUR_CSV_FILE_PATH"  # Path to save the CSV file
os.makedirs(os.path.dirname(csv_file), exist_ok=True)
with open(csv_file, 'w', newline='') as csvfile:
    header = ['data','psnr', 'ssim', 'fvd1', 'fvd2', 'fvd3']
    writer = csv.writer(csvfile)
    writer.writerow(header)

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# TEst for UCF101 Dataset
cover_videos, secret_videos = load_videos(data_dir,target_size=(240,320), max_frames=5)
max_batch = 1000
cover_videos = cover_videos[:max_batch] if cover_videos.shape[0] > max_batch else cover_videos
secret_videos = secret_videos[:max_batch] if secret_videos.shape[0] > max_batch else secret_videos
batch_size = 16
recon_cover, recon_secret = generate(model, cover_videos, secret_videos, batch_size, device)
psnr, ssim, fvd1, fvd2, fvd3 = calculate_metrics(cover_videos, recon_cover)
s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3 = calculate_metrics(secret_videos, recon_cover)
del cover_videos, secret_videos, recon_cover, recon_secret
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_dir, psnr, ssim, fvd1, fvd2, fvd3])
    writer.writerow([data_dir + '_secret', s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3])

print(f"Metrics for {data_dir}: PSNR={psnr}, SSIM={ssim}, FVD1={fvd1}, FVD2={fvd2}, FVD3={fvd3}")
print(f"Metrics for {data_dir}_secret: PSNR={s_psnr}, SSIM={s_ssim}, FVD1={s_fvd1}, FVD2={s_fvd2}, FVD3={s_fvd3}")

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for DAVIS Dataset
cover_videos, secret_videos = load_videos(data_dir,target_size=(480,640), max_frames=5)
max_batch = 500
cover_videos = cover_videos[:max_batch] if cover_videos.shape[0] > max_batch else cover_videos
secret_videos = secret_videos[:max_batch] if secret_videos.shape[0] > max_batch else secret_videos
batch_size = 4
recon_cover, recon_secret = generate(model, cover_videos, secret_videos, batch_size, device)
psnr, ssim, fvd1, fvd2, fvd3 = calculate_metrics(cover_videos, recon_cover)
s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3 = calculate_metrics(secret_videos, recon_cover)
del cover_videos, secret_videos, recon_cover, recon_secret
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_dir, psnr, ssim, fvd1, fvd2, fvd3])
    writer.writerow([data_dir + '_secret', s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3])

print(f"Metrics for {data_dir}: PSNR={psnr}, SSIM={ssim}, FVD1={fvd1}, FVD2={fvd2}, FVD3={fvd3}")
print(f"Metrics for {data_dir}_secret: PSNR={s_psnr}, SSIM={s_ssim}, FVD1={s_fvd1}, FVD2={s_fvd2}, FVD3={s_fvd3}")

In [ ]:
data_dir = "YOUR_VIDEO_DIR"  # Path to your video directory
# Test for MOT17 Dataset
cover_videos, secret_videos = load_videos(data_dir,target_size=(1080,1920), max_frames=5)
max_batch = 100
cover_videos = cover_videos[:max_batch] if cover_videos.shape[0] > max_batch else cover_videos
secret_videos = secret_videos[:max_batch] if secret_videos.shape[0] > max_batch else secret_videos
batch_size = 1
recon_cover, recon_secret = generate(model, cover_videos, secret_videos, batch_size, device)
psnr, ssim, fvd1, fvd2, fvd3 = calculate_metrics(cover_videos, recon_cover)
s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3 = calculate_metrics(secret_videos, recon_cover)
del cover_videos, secret_videos, recon_cover, recon_secret
with open(csv_file, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow([data_dir, psnr, ssim, fvd1, fvd2, fvd3])
    writer.writerow([data_dir + '_secret', s_psnr, s_ssim, s_fvd1, s_fvd2, s_fvd3])

print(f"Metrics for {data_dir}: PSNR={psnr}, SSIM={ssim}, FVD1={fvd1}, FVD2={fvd2}, FVD3={fvd3}")
print(f"Metrics for {data_dir}_secret: PSNR={s_psnr}, SSIM={s_ssim}, FVD1={s_fvd1}, FVD2={s_fvd2}, FVD3={s_fvd3}")